# 1. Business hypothesis and experiment overview

**Experiment:** 3.1_pretrained_similarity. **Method:** frozen shared text embeddings, normalised mean-history user profile, direct cosine ranking. Set the owner below.

**Hypothesis:** similarity between article content and past reading interests can recover known clicked articles in a catalogue-derived top-200 shortlist. The separate business re-ranker chooses the final five.

This notebook is **similarity only**. There is no logistic scorer, XGBoost, metadata weighting, classifier training, fine-tuning, or business re-ranking. It does not generate embeddings. It loads the group's saved Gemini text vectors; confirm the exact encoder and revision with the preprocessing owner.

The earlier hybrid scorer is archived locally under `Pretrained Models/hybrid_reference/` (ignored by Git), outside this active experiment. Its learned weights/checkpoints and click probabilities are not used here. Use a new similarity-only run ID rather than an old hybrid run folder.


# 2. Setup and run configuration

In Colab, enable a GPU if desired, mount Drive and upload the **new similarity-only `catalogue_pipeline.py`**. The upload option below accepts Colab's renamed filenames and explicitly imports the uploaded file, avoiding stale module copies. `DRIVE_ROOT` must point to the actual shared-folder shortcut in your account.

Start in audit mode. Switch to run mode after checking the paths and costs. Your run ID stays stable while switching phases; assign `RUN_ID = None` to start a new run before rerunning configuration. Completed runs can be reused without scoring again if their configuration is unchanged.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

EXPERIMENT_ID = '3.1_pretrained_similarity_gemini'
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    FORCE_UPLOAD = True  # Prompt again when updating the helper.
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    if FORCE_UPLOAD or not (CODE_DIR / 'catalogue_pipeline.py').exists():
        uploaded = files.upload()
        matches = [name for name in uploaded
                   if Path(name).stem.startswith('catalogue_pipeline')
                   and Path(name).suffix == '.py']
        if len(matches) != 1:
            raise ValueError(f'Select exactly one catalogue_pipeline Python file; received {list(uploaded)}')
        (CODE_DIR / 'catalogue_pipeline.py').write_bytes(uploaded[matches[0]])
else:
    candidates = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
    CODE_DIR = next((p for p in candidates if (p / 'catalogue_pipeline.py').exists()), None)
    if CODE_DIR is None:
        raise FileNotFoundError('Open from the repository root or this experiment folder.')

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy>=1.26', 'pandas>=2.2', 'pyarrow>=15',
                'scikit-learn>=1.4'], check=True)
import numpy as np
import pandas as pd

helper_path = CODE_DIR / 'catalogue_pipeline.py'
spec = importlib.util.spec_from_file_location('catalogue_pipeline', helper_path)
cp = importlib.util.module_from_spec(spec)
sys.modules['catalogue_pipeline'] = cp
spec.loader.exec_module(cp)
if getattr(cp, 'PIPELINE_VERSION', None) != 'similarity_top200_scores_v5':
    raise RuntimeError('Upload the new similarity-only helper, not the archived hybrid version.')
print('Loaded:', cp.__file__)


In [ ]:
OWNER = ''  # Your name for final handoff.
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
SEED = 42
RUN_PHASE = 'audit'  # 'audit' or 'run'
LOAD_RUN = None  # Optional completed similarity-only persistent run to reproduce.
previous_run_id = globals().get('RUN_ID')
RUN_ID = (previous_run_id if previous_run_id and '_sim_seed' in str(previous_run_id) else None) or (
    datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + f'_sim_seed{SEED}'
)
WORK = CODE_DIR / 'outputs' / RUN_ID
PERSISTENT_ROOT = DRIVE_ROOT / 'Experiments'
CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'owner': OWNER, 'run_id': RUN_ID,
    'drive_root': str(DRIVE_ROOT), 'seed': SEED, 'scope': 'development',
    'input_version': None,
    'embedding_source': {'provider': 'Gemini', 'model': None, 'revision': None},
    'embedding_column': 'text_embedding', 'embedding_dimension': None,
    'history_limit': None,  # All available past clicks; optionally an integer for latest N.
    'validation_requests': 100,  # Pilot; consider 1,000 fixed requests after measuring cost.
    'export_requests': {'train': 25, 'validation': 25, 'test': 25},
    'catalogue_batch_size': 4096,
    'similarity_device': 'auto',  # 'cpu', 'cuda', or automatic CUDA/CPU fallback.
    'similarity_min_gpu_batch': 256,
    'hash_inputs': False,
    'max_index_gb': 20, 'max_export_pairs': 10_000_000,
    'max_export_gb': 10, 'max_scoring_seconds': 3600,
    'persistent_root': str(PERSISTENT_ROOT),
    'paths': {},  # Override actual Drive input paths here if needed.
}
CONFIG['code_provenance'] = {
    'helper_sha256': hashlib.sha256(helper_path.read_bytes()).hexdigest(),
    'commit': None, 'uncommitted_changes': None,
}
try:
    CONFIG['code_provenance'].update(
        commit=subprocess.run(['git', '-C', str(CODE_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip(),
        uncommitted_changes=bool(subprocess.run(['git', '-C', str(CODE_DIR), 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout),
    )
except (subprocess.CalledProcessError, FileNotFoundError):
    pass
print('Drive root exists:', DRIVE_ROOT.exists())
print('Local run:', WORK)
print('No classifier training or checkpoint is required.')


# 3. Load and check shared data

Load the shared text embedding lookups, article IDs/publication times, fixed click histories, and original request/candidate keys. The engineered feature tables supply **only IDs, timestamps and genuine click labels for evaluation**. Feature vocabulary, metadata predictors and scaled/log variants are not required.

Article catalogue IDs and vector IDs are audited separately. Duplicate source keys, unreadable vectors, and temporal history violations raise errors. Test click labels, if present in an updated release, are ignored and are not loaded into the inference index; this policy is recorded in the audit. Original test impression IDs can repeat across users; the request key is `user_id:impression_id`.

Eligibility is unchanged: use the declared ordered catalogue union, retain the first snapshot per article ID, and require known publication time no later than the request. There is no extra age or access cutoff. Missing text vectors remain eligible with a documented zero score. Confirm the same catalogue/policy across the group before model comparison.


In [ ]:
PREPARED = cp.prepare(CONFIG, WORK)
audit = PREPARED['audit']
print('Pipeline:', audit['pipeline_version'])
print('Compute:', audit['compute'])
print('Catalogue articles:', audit['catalogue_articles'])
print('Catalogue articles missing vectors:', audit['catalogue_articles_without_vectors'])
print('Embedding dimension:', audit['embedding_info']['dimension'])
display(pd.DataFrame(audit['request_counts']).T)


In [ ]:
print('Planned development exports:')
display(pd.DataFrame(audit['planned_exports']).T[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
print('Full-split costs (not the default export scope):')
display(pd.DataFrame(audit['full_export_costs']).T[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
cp.check_cost(PREPARED)


# 4. Prepare model inputs

For each user, take their eligible past-click article embeddings. Each article vector is L2-normalised; average the selected vectors and normalise that mean to form the user profile. Repeated clicks retain repeated weight. If `history_limit` is set, select the latest N actual clicks by timestamp before dropping missing vectors.

History is a fixed snapshot strictly before the first shared request in each split. It is not updated with current behavior labels. Missing history, no available vectors, or a cancelled mean gives no profile. Such requests receive zero scores with deterministic article-ID ties, and are counted explicitly.

Cosine similarity alone ranks candidates. The availability/coverage flags are diagnostics, not inputs to a learned model. Candidate metadata, current/subsequent read outcomes and click labels do not change scores.


In [ ]:
print('Scoring rule: score(user, article) = normalised_article_vector @ normalised_user_profile')
print('History setting:', CONFIG['history_limit'] or 'all eligible past clicks')
print('Fallback: score 0 for unavailable profile or candidate vector; retain eligible articles.')


# 5. Define and train / load model

**There is nothing to train.** No epochs, alpha trials, classifier checkpoints, scalers or learned feature weights are used. This section saves a small JSON inference specification or loads the same specification from a completed similarity-only run. Old hybrid `selected.joblib` files are incompatible and are not read.

Set `RUN_PHASE='run'` after the audit. Rerun sections 3-4 if changing configuration. Use a fresh run ID for a different encoder, text variant or history setting. No `cp.train()` call is needed.


In [ ]:
MODEL = None
if PREPARED['work'] != WORK or PREPARED['config'] != CONFIG:
    raise RuntimeError('Rerun sections 3-4 after changing configuration.')
if RUN_PHASE == 'run':
    MODEL = cp.load_run(PREPARED, LOAD_RUN) if LOAD_RUN else cp.define_model(PREPARED)
    print(json.dumps(MODEL, indent=2))
else:
    print('Audit only. Set RUN_PHASE="run" after reviewing costs.')


# 6. Validation diagnostics and explainability

Evaluate the fixed validation-request sample against its **full eligible catalogue**. Primary metric: mean known-click Recall@200. Secondary: observed-click nDCG@200. Keep excluded original known clicks in both denominators/ideals. Empty pools with known clicks receive zero. Unshown outcomes remain unknown, not negative training labels.

There is no automatic tuning search in this notebook. If you later compare history settings, encoders or text variants, fix the same request IDs and select using validation Recall@200, then nDCG@200. Never select using test outcomes. Previously inspected/repeated validation is development evidence, not an untouched final test.


In [ ]:
VALIDATION = None
if RUN_PHASE == 'run':
    VALIDATION = cp.evaluate(PREPARED, MODEL)
    display(pd.json_normalize(VALIDATION['metrics']))
    print('Scoring seconds:', round(VALIDATION['seconds'], 2))
    print('Catalogue pairs scored:', VALIDATION['eligible_pairs'])
    print('History coverage:', VALIDATION['history_coverage'])


Export only `rankings/{split}/submission_top_200.csv`, with **behaviour_id,article_ids,similarity_scores**. Each row contains a JSON list of up to 200 original article IDs, ordered by cosine similarity. The `similarity_scores` JSON list matches the IDs position by position. Every eligible article is scored in memory. Full catalogue scores and individual recorded-slate scores are not saved. Per-request eligibility audits and aggregate metrics are retained. Saved shortlists are checked by recomputing the rankings. No business reranking is applied.

Use a fresh run ID with this version. The manifest labels this as a top-200-only handoff, which differs from the original shared full-score contract.


In [ ]:
RESULT = None
if RUN_PHASE == 'run':
    RESULT = cp.execute(PREPARED, MODEL)
    display(pd.json_normalize(RESULT['metrics']))
    print('Exported-validation metrics:')
    display(pd.json_normalize(RESULT['exported_validation']))
    print('Export checks:', RESULT['checks'])
    print('Complete final handoff:', RESULT['handoff_complete'])


# 8. Reload check

The full run bundle is copied to shared Drive, with the JSON inference model, code, source versions and a score probe. The helper reloads code/configuration from that persistent copy, reopens the versioned text embedding tables, recomputes probe cosine scores and checks agreement within 2e-6. No classifier is fitted or loaded. Temporary SQLite indexes are rebuilt from the recorded source files when needed rather than being part of the handoff model.

Partial/failed persistence is recorded as incomplete. Missing real-data access cannot be replaced by synthetic performance results. CUDA is optional and its hardware parity check is skipped locally when no CUDA hardware is available.


In [ ]:
if RESULT is not None:
    print(json.dumps(RESULT['persistence'], indent=2))
    if RESULT['persistence']['persistent_copy_complete']:
        saved = Path(RESULT['persistence']['persistent_location'])
        print('Saved fixed inference model:', saved / 'model/model_config.json')
        print('No learned model weights/checkpoint are needed.')


# 9. Business interpretation and handoff

Report whether direct pretrained content similarity recovers known clicked news in the catalogue top 200, the effect of unavailable profiles/vectors, observed-click ranking metrics, compute and storage cost. The Stage 2 owner receives the top-200 IDs plus the article/user information needed for the separate freshness/diversity/continuity re-ranker.

A top-200 similarity score is not click probability and does not establish retention gains. Do not compare these catalogue results directly with older recorded-slate hybrid results. The shared Drive data has not been run in the local coding environment; implementation verification uses synthetic fixtures only. Run the notebook on the actual shared files before reporting project metrics or a complete persistent handoff.


In [ ]:
if RESULT is not None:
    example = json.loads((WORK / 'validation_example.json').read_text())
    display(pd.DataFrame(example['articles']))
    print('Validation request:', example['behaviour_id'])
    print(example['note'])
    print('Bundle:', RESULT['persistence'].get('persistent_location', str(WORK)))
    print('Test outcome metrics: unavailable.')
